# 강의 03 · 실습 1 — 에이전트 동작 원리 · (4) 고난도 I

## 1. 문제상황

- 구름월드 고객센터 안내 프로그램(FAQ 조회 도구와 현재 시각 도구를 쥔 도구 호출 루프)을 하루 동안 시험 운영했습니다.
- FAQ에 있는 운영시간·주차·환불 질문은 잘 답했지만, 「반려동물 데리고 들어갈 수 있나요」처럼 FAQ에 없는 항목을 묻자 프로그램이 오류 화면을 내며 멈췄습니다.
- 도구 faq_lookup이 없는 항목을 받으면 KeyError를 내고, 그 예외가 루프 밖으로 튀어나와 실행 전체가 끊긴 탓입니다.
- 멈춘 뒤에 들어온 주차 질문과 환불 질문은 답을 받지 못했습니다.
- 담당자는 프로그램이 멈추는 대신 「그 항목은 안내할 수 없다」고 답하고, 다음 질문을 계속 받기를 원합니다.

## 2. 문제와 목표

- **문제**: 도구 실행 중 예외가 나면 루프가 끊기고, 모델은 도구가 실패했다는 사실을 알 길이 없습니다. 실패한 질문 하나가 뒤의 질문까지 막습니다.
- **목표**
  - 도구 실행 오류를 예외로 던지지 않고 오류 문자열로 바꿔 대화 기록에 되먹여야 합니다.
    - 오류 문자열: 「도구 실행 오류: <예외 이름>: <내용>」
  - 모델이 오류를 읽고 다음 행동을 정하게 하는 루프를 만듭니다.
    - 다음 행동: 안내 불가 답변, 또는 다른 항목 조회
- **목표 달성 여부의 판정 기준**:
  - FAQ에 없는 항목 질문에서 `[도구 결과] 도구 실행 오류: KeyError: '반려동물'`이 찍힌 뒤 실행이 끊기지 않고 최종 답이 나오며,
  - 이어지는 주차 질문과 환불 질문도 도구 결과를 반영한 답으로 끝나는 것을 실행 기록에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec03_ex01_s4_diagram.svg)

## 4. 단계별 요구사항

1. **도구를 선언합니다.**
    - FAQ 사전(운영시간·주차·환불)에서 항목을 조회하는 `faq_lookup(topic)`과 현재 날짜·시각을 돌려주는 `get_now()`를 `@tool`로 선언합니다.
    - FAQ 사전의 값은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
    - `faq_lookup`은 없는 항목에 `KeyError`를 그대로 냅니다(도구 안에서 잡지 않습니다). 도구 설명은 「시설 이용에 관한 질문이면 먼저 이 도구로 항목을 찾아본다」로 적어, 모델이 FAQ에 있는지 모르는 항목도 조회를 시도하게 합니다.
2. **도구를 모델에 묶습니다.**
    - `bind_tools`로 두 도구를 모델에 붙여 도구를 쥔 모델 `llm_tools`를 만들고, 도구 이름으로 도구 객체를 찾는 사전 `TOOLS`를 만듭니다.
3. **모델을 호출하고 도구 호출 요청을 판정합니다.**
    - 사용자 질문을 `HumanMessage`로 담은 대화 기록을 `llm_tools`에 넣어 호출하고, 돌아온 응답의 `tool_calls`가 비어 있는지로 도구 호출 요청 여부를 판정합니다.
    - 「반려동물 데리고 들어갈 수 있나요?」를 첫 질문으로 넣어 첫 호출의 `tool_calls`에 `faq_lookup` 요청이 실리는지 확인합니다.
4. **도구 결과와 오류를 되먹여 반복합니다.**
    - 도구 실행을 `try`/`except`로 감싸고, 예외가 나면 `f"도구 실행 오류: {type(e).__name__}: {e}"` 문자열을 결과로 삼습니다.
    - 성공 결과와 오류 문자열을 똑같이 `ToolMessage`로 대화 기록에 붙이고 모델을 다시 호출합니다.
    - 반복 상한은 4회입니다.
5. **세 질문으로 실행합니다.**
    - 「반려동물 데리고 들어갈 수 있나요?」, 「주차는 얼마인가요?」, 「환불은 언제까지 되나요?」를 차례로 넣어, 오류가 난 질문 뒤에도 다음 질문이 정상 처리되는 것을 출력으로 보입니다.
    - 출력 줄의 이름은 「[도구 호출]」「[도구 결과]」「[최종 답]」이고, 질문마다 「=== N번 질문 ===」 줄로 시작합니다.

## 5. 코드 골격 — 도구 호출 루프 4단

네 단계는 같고, ④ 결과 되먹임 단계 안에서 오류를 결과 문자열로 바꾸는 처리가 더해집니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 도구 선언 | 파이썬 함수 위에 표시 한 줄을 붙여 도구로 만듭니다 | `@tool(parse_docstring=True)` | 1 |
| ② 도구 묶기 | 도구 목록을 모델에 붙여 도구를 쥔 모델을 만듭니다 | `llm.bind_tools([...])` | 2 |
| ③ 반복 호출·판정 | 대화 기록을 넣어 호출하고, 도구 호출이 실렸는지 봅니다 | `res.tool_calls` | 3 |
| ④ 결과 되먹임 | 도구를 실행하고, 성공 결과 또는 오류 문자열을 대화 기록에 붙여 다시 호출합니다 | `try`/`except`, `ToolMessage(content=..., tool_call_id=...)` | 4, 5 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [1]:
import os

from datetime import datetime
from dotenv import load_dotenv, find_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
print("모델 준비를 마쳤습니다.")

# 주어진 자료
FAQ = {
    "운영시간": "매일 09:30~21:00에 운영합니다.",
    "주차": "주차장은 4,000대 규모이며 최초 30분은 무료입니다.",
    "환불": "이용일 전날까지 전액 환불, 당일은 50% 환불입니다.",
}


모델 준비를 마쳤습니다.


### 단계 ① — 도구 선언 (요구사항 1)

- 도구의 실체는 파이썬 함수입니다. 함수 위에 `@tool`을 붙이면 함수 이름·독스트링·인자 타입에서 모델에게 보여 줄 도구 설명(스키마)이 만들어집니다.
- `parse_docstring=True`는 독스트링의 `Args:` 항목을 인자 설명으로 씁니다. 모델은 도구 설명과 인자 설명을 보고 어느 도구를 언제 부를지 정합니다.
- `get_now`는 인자가 없으므로 `@tool`만 붙입니다.

In [2]:
# 여기에 단계 ①(도구 두 개 선언 — 도구 설명은 요구사항 1대로)을 작성합니다.

@tool(parse_docstring=True)
def faq_lookup(topic: str) -> str:
    """구름월드 FAQ에서 항목을 조회한다. 시설 이용에 관한 질문이면 먼저 이 도구로 항목을 찾아본다.
 
    Args:
        topic: 조회할 항목 이름. 예: 운영시간, 주차, 환불
    """
    return FAQ[topic]
 
 
@tool
def get_now() -> str:
    """현재 날짜와 시각을 돌려준다."""
    return datetime.now().strftime("%Y-%m-%d %H:%M")
 
 
for t in [faq_lookup, get_now]:
    print(f"[도구] {t.name}: {t.description} / 인자: {list(t.args)}")

[도구] faq_lookup: 구름월드 FAQ에서 항목을 조회한다. 시설 이용에 관한 질문이면 먼저 이 도구로 항목을 찾아본다. / 인자: ['topic']
[도구] get_now: 현재 날짜와 시각을 돌려준다. / 인자: []


### 단계 ② — 도구 묶기 (요구사항 2)

- `bind_tools`는 도구 목록을 모델에 붙여, 호출할 때마다 도구 설명을 함께 보내는 새 모델 객체를 돌려줍니다. 원래의 `llm`은 바뀌지 않습니다.
- `TOOLS`는 모델이 보낸 도구 이름을 실제 도구 객체로 바꾸는 사전입니다. 단계 ④에서 도구를 실행할 때 씁니다.

In [3]:
# 여기에 단계 ②(도구 묶기와 TOOLS 사전)를 작성합니다.
llm_tools = llm.bind_tools([faq_lookup, get_now])
TOOLS = {t.name: t for t in [faq_lookup, get_now]}

print("모델에 묶인 도구:", list(TOOLS))

모델에 묶인 도구: ['faq_lookup', 'get_now']


### 단계 ③ — 반복 호출·판정 (요구사항 3)

- 대화 기록은 메시지 객체의 리스트입니다. 첫 항목은 사용자 질문을 담은 `HumanMessage`입니다.
- 도구를 쥔 모델을 호출하면 `AIMessage`가 돌아옵니다. 모델이 도구를 부르기로 정했으면 `tool_calls`에 도구 이름·인자·호출 id가 실립니다. `tool_calls`가 비어 있으면 그 응답이 최종 답입니다.
- 아래 셀은 FAQ에 없는 항목 질문의 첫 호출 응답을 열어 봅니다. 도구 설명이 「먼저 찾아본다」이므로 모델은 없는 항목에도 `faq_lookup`을 요청합니다. 인자는 처음부터 딕셔너리로 돌아오므로 문자열 파싱이 필요 없습니다.

In [4]:
# 여기에 단계 ③(첫 호출과 tool_calls 판정)을 작성합니다.
messages = [HumanMessage(content="운영시간이 어떻게 되나요?")]
res = llm_tools.invoke(messages)

print("응답 종류:", type(res).__name__)
print("tool_calls:", res.tool_calls)
if res.tool_calls:
    print("판정: 도구 호출 요청이 있습니다. 도구 실행으로 갑니다.")
else:
    print("판정: 도구 호출 요청이 없습니다. 최종 답입니다.")

응답 종류: AIMessage
tool_calls: [{'name': 'faq_lookup', 'args': {'topic': '운영시간'}, 'id': 'call_perTCqgfgbp9YC7nifiu9p9G', 'type': 'tool_call'}]
판정: 도구 호출 요청이 있습니다. 도구 실행으로 갑니다.


### 단계 ④ — 결과 되먹임 (요구사항 4, 5)

- 도구 실행을 `try`/`except`로 감쌉니다. 예외가 나면 예외 이름과 내용을 담은 문자열을 만들어 결과 위치에 넣습니다.
- 성공 결과와 오류 문자열은 모델 입장에서 똑같은 「도구 결과」입니다. 둘 다 `ToolMessage`로 대화 기록에 붙이고 모델을 다시 호출합니다.
- 모델은 오류 문자열을 읽고 다른 항목을 조회하거나 안내할 수 없다고 답합니다. 어느 쪽이든 루프는 끊기지 않습니다.

In [20]:
# 여기에 단계 ④(오류를 되먹이는 run_agent 함수와 세 질문 실행)를 작성합니다.

def run_agent(question: str, max_turn: int = 4) -> str:
    messages = [HumanMessage(question)]
    print(messages)
    for _ in range(max_turn):
        res = llm_tools.invoke(messages)
        if not res.tool_calls:
            return res.content
        messages.append(res)
        print(messages)
        for call in res.tool_calls:
            try :    
                result = TOOLS[call["name"]].invoke(call["args"])
            except Exception as e:
                result = f"도구 실행 오류 :{type(e).__name__} :{e}"
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
    return "반복 한도 초과"


QUESTIONS = [
    "운영시간이 어떻게 되나요?",
    "지금 몇 시인가요?",
    "애완동물이랑 탈수있는 놀이기구는 뭔가요",
]

for i in (QUESTIONS):
    print(f"=== {i} ===")
    answer = run_agent(i)
    print(f"  [최종 답] {answer}")
    print()

=== 운영시간이 어떻게 되나요? ===
[HumanMessage(content='운영시간이 어떻게 되나요?', additional_kwargs={}, response_metadata={})]
[HumanMessage(content='운영시간이 어떻게 되나요?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'tool_calls': [ChatCompletionMessageToolCall(index=0, function=Function(arguments='{"topic":"운영시간"}', name='faq_lookup'), id='call_qC22XIWjuVXGs6KDkealCKw6', type='function')]}, response_metadata={'token_usage': Usage(completion_tokens=35, prompt_tokens=110, total_tokens=145, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=12, rejected_prediction_tokens=None, text_tokens=None, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cache_write_tokens=0, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0)), 'model': 'openai/gpt-5.6-luna', 'finish_reason': 'tool_calls', 'logprobs': None

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. 1번 질문(반려동물)에서 `[도구 호출] faq_lookup {'topic': '반려동물'}` 뒤에 `[도구 결과] 도구 실행 오류: KeyError: '반려동물'`이 찍히고, 실행이 끊기지 않은 채 최종 답이 나옵니다. 최종 답은 안내할 수 없다는 내용이거나 다른 항목을 조회한 결과입니다.
2. 2번 질문(주차)과 3번 질문(환불)에서 `faq_lookup`이 정상 결과를 돌려주고 최종 답에 그 내용이 들어 있습니다.
3. 세 질문이 모두 `=== N번 질문 ===` 줄로 시작해 `[최종 답]` 줄로 끝납니다. 오류가 난 질문이 뒤의 질문을 막지 않았다는 뜻입니다.

세 가지가 모두 확인되면 완성입니다.